# Explainable AI on COVID-19 Chest X-Ray Data

##### By Juan Mallo de la Calle and Jesús Rincón Laguarta

> **Note on repository version:** notebook outputs have been cleared in the public version to avoid redistributing chest X-ray images or Grad-CAM overlays from the external dataset. To reproduce the figures, download the dataset locally and run the notebook.

## Table of Contents

- [1. Introduction: Why Explainability Matters in COVID-19 X-ray AI](#1.-Introduction:-Why-Explainability-Matters-in-COVID-19-X-ray-AI)
- [2. Literature Review: Explainability in COVID-19 X-ray Models](#2.-Literature-Review:-Explainability-in-COVID-19-X-ray-Models)
  - [2.1 XAI in High-Performance CNNs](#2.1-XAI-in-High-Performance-CNNs)
  - [2.2 Grad-CAM as the Standard Visual Explanation](#2.2-Grad-CAM-as-the-Standard-Visual-Explanation)
  - [2.3 LIME and SHAP for Model-Agnostic Interpretability](#2.3-LIME-and-SHAP-for-Model-Agnostic-Interpretability)
  - [2.4 XAI for Bias Detection and Preprocessing Design](#2.4-XAI-for-Bias-Detection-and-Preprocessing-Design)
  - [2.5 Multi-Model and Multi-Modal Explainable Systems](#2.5-Multi-Model-and-Multi-Modal-Explainable-Systems)
- [3. Project Development](#3.-Project-Development)
  - [3.1 Dataset Definition: COVID-19 Chest X-Ray Dataset (Cohen et al.)](#3.1-Dataset-Definition:-COVID-19-Chest-X-Ray-Dataset-(Cohen-et-al.))
  - [3.2 Setup and Initial Data Inspection](#3.2-Setup-and-Initial-Data-Inspection)
  - [3.3 Preprocessing](#3.3-Preprocessing)
    - [3.3.1 Tabular Preprocessing](#3.3.1-Tabular-Preprocessing)
    - [3.3.2 Image Preprocessing](#3.3.2-Image-Preprocessing)
  - [3.4 Exploratory Data Analysis (EDA)](#3.4-Exploratory-Data-Analysis-(EDA))
    - [3.4.1 Age Distribution by Class](#3.4.1-Age-Distribution-by-Class)
    - [3.4.2 Sex Distribution within Each Class](#3.4.2-Sex-Distribution-within-Each-Class)
    - [3.4.3 View Type Proportions](#3.4.3-View-Type-Proportions)
    - [3.4.4 Class Imbalance](#3.4.4-Class-Imbalance)
    - [3.4.5 Images per Patient](#3.4.5-Images-per-Patient)
    - [3.4.6 Word Cloud for Clinical Notes per Class (Diagnosis)](#3.4.6-Word-Cloud-for-Clinical-Notes-per-Class-(Diagnosis))
  - [3.5 Patient-Level Transformation](#3.5-Patient-Level-Transformation)
    - [3.5.1 Label Transition Statistics](#3.5.1-Label-Transition-Statistics)
    - [3.5.2 Sex Distribution Among Patients with Diagnosis Transitions](#3.5.2-Sex-Distribution-Among-Patients-with-Diagnosis-Transitions)
    - [3.5.3 Age Distribution per Diagnosis](#3.5.3-Age-Distribution-per-Diagnosis)
  - [3.6 Training with Tabular Data](#3.6-Training-with-Tabular-Data)
    - [3.6.1 Cleaning and Preprocessing](#3.6.1-Cleaning-and-Preprocessing)
    - [3.6.2 Random Forest Classifier](#3.6.2-Random-Forest-Classifier)
    - [3.6.3 Feature Importance](#3.6.3-Feature-Importance)
    - [3.6.4 XGBoost](#3.6.4-XGBoost)
    - [3.6.5 SHAP and LIME](#3.6.5-SHAP-and-LIME)
  - [3.7 Training with Images](#3.7-Training-with-Images)
    - [3.7.1 Data Preparation](#3.7.1-Data-Preparation)
    - [3.7.2 Train/Val/Test Split](#3.7.2-Train/Val/Test-Split)
    - [3.7.3 Model Definition: Simple CNN](#3.7.3-Model-Definition:-Simple-CNN)
    - [3.7.4 Training and Evaluation](#3.7.4-Training-and-Evaluation)
    - [3.7.5 Grad-CAM for CNN Interpretability](#3.7.5-Grad-CAM-for-CNN-Interpretability)
  - [3.8 Explainability with Pretrained X-Ray Models](#3.8-Explainability-with-Pretrained-X-Ray-Models)
    - [3.8.1 Grad-CAM Visualization with TorchXRayVision](#3.8.1-Grad-CAM-Visualization-with-TorchXRayVision)
    - [3.8.2 Grad-CAM Across Layers](#3.8.2-Grad-CAM-Across-Layers)
- [4. Conclusions](#4.-Conclusions)
- [5. Future Work](#5.-Future-Work)
- [6. Demo](#6.-Demo)

## 1. Introduction: Why Explainability Matters in COVID-19 X-ray AI

The COVID-19 pandemic accelerated the development of AI models for chest X-ray (CXR) diagnosis. However, their clinical adoption depends heavily on explainability, to ensure models make decisions for the right medical reasons and gain clinician trust.

Recent studies have emphasized that combining high-performing CNN classifiers with explainable AI (XAI) methods like Grad-CAM, LIME, and SHAP is now a standard practice in COVID-19 CXR analysis.

Two recent studies highlight the growing importance of explainability in AI-based COVID-19 diagnosis:

- A 2023 study published in *Heliyon* [ScienceDirect, 2023](https://www.sciencedirect.com/science/article/pii/S2405844023023447) applied Grad-CAM and LIME across multiple CNN architectures, demonstrating that combining XAI techniques leads to more robust interpretations and significantly improves clinician confidence in model outputs.

- A 2024 article in *Scientific Reports* [Nature, 2024](https://www.nature.com/articles/s41598-024-75915-y) introduced an improved Grad-CAM++ method for sharper and more clinically meaningful visual explanations. Radiologist evaluations confirmed that these explanations aligned well with COVID-19 pathology and proved useful for model refinement and debugging.

## 2. Literature Review: Explainability in COVID-19 X-ray Models

In the last 3–5 years, research in XAI for COVID-19 chest X-rays has focused on:

### 2.1 XAI in High-Performance CNNs

State-of-the-art CNNs (often pretrained on ImageNet or large medical datasets) reach >90% accuracy when detecting COVID-19. These models are now routinely combined with XAI techniques to validate whether they rely on medically relevant features. Studies show that Grad-CAM outputs often align with regions of lung opacity, reinforcing trust in AI predictions.

### 2.2 Grad-CAM as the Standard Visual Explanation

Grad-CAM highlights the areas in the lungs that most influenced a classification (e.g., ground-glass opacities in COVID-19). Compared to LIME or SHAP, it is faster, more intuitive and widely adopted by clinicians.

### 2.3 LIME and SHAP for Model-Agnostic Interpretability

LIME segments the image into superpixels and perturbs them to reveal their impact. SHAP assigns importance scores to pixels/patches based on Shapley values. Though more computationally expensive, these techniques complement Grad-CAM by providing layer-independent and sometimes more granular insights.

### 2.4 XAI for Bias Detection and Preprocessing Design

XAI tools have revealed biases in early COVID-19 models—many inadvertently focused on hospital labels, machine annotations or tubes, not actual lung pathology. These findings prompted the adoption of preprocessing steps such as lung masking, image cropping and standardization of views.

### 2.5 Multi-Model and Multi-Modal Explainable Systems

Recent studies employ ensembles (e.g., DenseNet + ResNet + VGG) and fuse image features with clinical data (e.g., CRP, ferritin). These models use XAI on both modalities (e.g., Grad-CAM for images, SHAP for labs), providing transparent multi-source reasoning that aligns with medical expectations.

##### Our Approach

Following the state of the art, our project:

- Preprocesses images (resize, greyscale, artifact handling) to minimize bias.
- Trains a CNN classifier for COVID-19 detection.
- Applies Grad-CAM and additional XAI methods to validate whether the model attends to clinically meaningful regions.
- Uses XAI findings to refine preprocessing and improve model trustworthiness.

## 3. Project Development

This project corresponds to the course **Large Scale Media Analytics** and focuses on **Explainable AI (XAI)** on tabular and image data of Covid-19.

### 3.1 Dataset Definition: COVID-19 Chest X-Ray Dataset (Cohen et al.)

The **COVID-19 Chest X-Ray Dataset** compiled by Cohen et al. ([GitHub repository](https://github.com/ieee8023/covid-chestxray-dataset)), is a public collection of chest radiographs gathered during the COVID-19 pandemic. It contains primarily frontal X-rays of COVID-19 patients, with additional cases of other pneumonias and a limited number of normal (healthy) images. Each image is accompanied by rich metadata including patient demographics (age, sex), clinical findings, RT-PCR confirmation, outcomes like ICU admission, and occasionally laboratory values.

The dataset is heterogeneous, combining images from different hospitals, countries, and acquisition settings. While it has been essential for early AI research, it is biased toward severe hospitalized cases and suffers from class imbalance, with COVID-19 images vastly outnumbering healthy controls. Artifacts such as hospital markings or medical devices are common and can confound models if not properly addressed.

To work effectively with this dataset, standard preprocessing is required, including resizing images, normalizing intensities, handling missing metadata and ensuring patient-wise train/test splits. Despite its limitations, this dataset remains a cornerstone for explainable AI (XAI) research in COVID-19 imaging.

### 3.2 Setup and Initial Data Inspection

In this section, we import all necessary libraries and load the COVID-19 Chest X-Ray metadata. We begin by organizing our environment and inspecting the raw data structure to spot any glaring issues (e.g., missing values, unexpected types) before proceeding to more in-depth analysis.


In [ ]:
# Standard Libraries
import os
import re
from pathlib import Path
from collections import defaultdict

# Scientific Computing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud

# Image Processing
import cv2
from PIL import Image

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# PyTorch & TorchVision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# TorchCAM & X-Ray Vision
from torchcam.methods import GradCAM
import torchxrayvision as xrv

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# XGBoost
import xgboost as xgb

# XAI
import shap
from lime.lime_tabular import LimeTabularExplainer

# Ignore deprecation and experimental warnings
import warnings
from cryptography.utils import CryptographyDeprecationWarning

warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)
warnings.filterwarnings("ignore", message="Using `tqdm.autonotebook.tqdm` in notebook mode")
# Project paths
# Run this notebook from the repository root. If your environment starts inside
# the notebooks/ folder, the fallback path below will still work.
DATA_DIR = Path("data/covid-chestxray-dataset")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data/covid-chestxray-dataset")

METADATA_PATH = DATA_DIR / "metadata.csv"
RAW_IMAGE_DIR = DATA_DIR / "images"
PREPROC_IMAGE_DIR = DATA_DIR / "images_preproc"
PREPROC_METADATA_PATH = DATA_DIR / "metadata_preproc.csv"

ARTIFACT_ANNOTATIONS_PATH = Path("data/artifact_annotations.csv")
if not ARTIFACT_ANNOTATIONS_PATH.exists():
    ARTIFACT_ANNOTATIONS_PATH = Path("../data/artifact_annotations.csv")


In [ ]:
# Load metadata CSV
metadata = pd.read_csv(METADATA_PATH)

# Display dataset dimensions and first few rows
print("Shape:", metadata.shape)
print("\nHead:\n", metadata.head())


The dataset contains **950 rows** (one per image) and **30 columns** of metadata. The first few rows show patient demographics (`patientid`, `sex`, `age`), diagnostic labels (`finding`, `RT_PCR_positive`), and image file references (`filename`, `url`).

### 3.3 Preprocessing

#### 3.3.1 Tabular Preprocessing

The repository includes a small set of CT volumes under `volumes/`. Dropping these entries ensures that every `filename` refers to a valid 2D X-ray in `images/`.


In [ ]:
# Remove entries from the 'volumes' folder
metadata = metadata[metadata['folder'] != 'volumes'].reset_index(drop=True)

# Filtrar solo X-ray
metadata = metadata[metadata['modality'] == 'X-ray'].reset_index(drop=True)

print(f"Remaining samples: {len(metadata)}")
print(metadata['folder'].value_counts())


With this clean set of image records, we can proceed into class counts, missing-value analysis and all EDA steps.

##### Label Distribution

Knowing how many images belong to each diagnostic category informs both our modeling strategy (e.g. whether to group rare labels, oversample, or focus on a subset) and our split/augmentation design. Thus, we observe the class distribution:


In [ ]:
# Count of each finding
label_counts = metadata['finding'].value_counts()
print("Top 10 findings:\n", label_counts.head(10))


In [ ]:
# Bar plot of the top 10 findings
plt.figure(figsize=(12,6))
label_counts.head(10).plot.bar(color='skyblue')
plt.title('Top 10 Diagnostic Findings in COVID-19 CXR Dataset')
plt.ylabel('Number of Images')
plt.xlabel('Finding')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


Pneumonia/Viral/COVID-19 accounts for 504 images (~61 %). The next most frequent label, “todo”, has 83 images but lacks clinical specificity and will need to be reviewed or excluded. Healthy cases (“No Finding”) are extremely under-represented (18 images). Many diagnostic subtypes (e.g. MERS, SARS, fungal) occur in single‐digit counts.

Such a skewed distribution motivates grouping rare subtypes into an “Other Pneumonia” category, focusing on a simplified 3-class task (COVID-19 vs. Other Pneumonia vs. Normal) and planning class weights or oversampling during training.

Next, we will compute the number of unique patients. Ensuring that our train/val/test split is performed by patient (and not by image) prevents information leakage from multiple X-rays of the same individual.


In [ ]:
# Count unique patients
n_patients = metadata['patientid'].nunique()
print(f"Number of unique patients: {n_patients}")


There are 449 distinct patients in the dataset. On average, each patient contributes ~2 images (866 images / 449 patients), so a patient‐wise split is essential. To ensure no single patient dominates the dataset, we check:


In [ ]:
counts = metadata['patientid'].value_counts()
display(counts.to_frame("image_count"))


##### Handling Missing‐Value

In the study of missing values, we observed many of the clinical and laboratory columns have missing entries. We need to decide which features are too sparse to include and which require imputation.


In [ ]:
# Count missing values
missing_counts = metadata.isnull().sum().sort_values(ascending=False)
print("Missing values by column:\n", missing_counts)


In [ ]:
# Bar chart of missingness
plt.figure(figsize=(12,6))
missing_counts.plot.bar(color='lightcoral')
plt.title('Missing Values per Metadata Column')
plt.ylabel('Number of Missing Entries')
plt.xlabel('Column')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()


Columns such as leukocyte_count, lymphocyte_count, extubated have >90 % missing data, these will likely be excluded or require careful imputation if retained. Moreover, demographics (age, sex) are reasonably populated (~75 % non‐null) and RT_PCR_positive has ~38 % recorded values, making it a noisy but potentially informative feature after mapping and imputation.

Many metadata fields in the COVID-19 Chest X-ray dataset are highly incomplete. To ensure robustness and simplify downstream processing, we remove all columns with more than 50% missing values.

This threshold allows us to retain partially complete but useful features (e.g., age, sex, RT-PCR status), while discarding sparse fields such as `leukocyte_count`, `extubated`, or free-text columns like `other_notes`. Dropping these fields prevents potential bias or noise from incomplete data.


In [ ]:
# Drop columns with >50% missing values
threshold = 0.5
missing_fraction = metadata.isnull().mean()

# Identify and drop sparse columns
cols_to_drop = missing_fraction[missing_fraction > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns due to >55% missing values:")
print(cols_to_drop)

metadata = metadata.drop(columns=cols_to_drop)

# Check resulting shape
print(f"Remaining columns: {metadata.shape[1]}")


In [ ]:
# Count missing values
missing_counts = metadata.isnull().sum().sort_values(ascending=False)
print("Missing values by column:\n", missing_counts)


In addition, we drop several metadata fields that do not contribute meaningful information to our analysis or modeling process:

- `url`: External reference link; redundant and not needed for modeling.
- `modality`: Constant across samples (all X-rays); not informative.
- `location`: Hospital or country of origin; potentially introduces confounding and not used in current analysis.
- `license`: Legal metadata, irrelevant for our tasks.

Removing these columns simplifies the dataset and focuses our attention on clinically or technically relevant features.


In [ ]:
# Drop columns that do not add value to modeling or analysis
cols_to_remove = ['url', 'modality', 'location', 'license']
metadata.drop(columns=cols_to_remove, inplace=True)

# Check new shape
print(f"Remaining columns after dropping: {metadata.shape[1]}")


After cleaning irrelevant columns, we proceed to handle missing values for the remaining features.

We apply context-aware imputation strategies:
- `RT_PCR_positive`: filled as `'Unclear'` to reflect clinical uncertainty.
- `date`: parsed into datetime; missing values left as `NaT`.
- `offset`: filled with `0.0` assuming it corresponds to the reference day.
- `age`: imputed with the dataset's median age.
- `clinical_notes`: filled with an empty string to preserve text format.
- `sex`: imputed with `'Unknown'` to reflect missing demographic info.

This ensures the dataset is ready for downstream processing while maintaining clinical interpretability.


In [ ]:
# Fill RT_PCR_positive with 'Unclear' where missing
metadata['RT_PCR_positive'] = metadata['RT_PCR_positive'].fillna('Unclear')

# Parse and keep date as datetime — leave missing as NaT
metadata['date'] = pd.to_datetime(metadata['date'], errors='coerce')

# Fill offset with 0.0
metadata['offset'] = metadata['offset'].fillna(0.0)

# Fill age with median
median_age = metadata['age'].median()
metadata['age'] = metadata['age'].fillna(median_age)

# Fill clinical_notes with empty string
metadata['clinical_notes'] = metadata['clinical_notes'].fillna('')

# Fill sex with 'Unknown'
metadata['sex'] = metadata['sex'].fillna('Unknown')


In addition, we identify if certain views are over-represented or need filtering:


In [ ]:
view_counts = metadata['view'].value_counts()
print(view_counts)


Chest X-rays can be acquired from different anatomical perspectives, most commonly PA (posteroanterior), AP (anteroposterior), and AP Supine. These views differ in clinical context and visual characteristics:
- PA is typically taken in standing position and is considered the most standard view for diagnosis.
- AP and AP Supine are more common in hospital settings (e.g., bedridden or ICU patients), and often show magnified or distorted heart/lung structures.
- Views like Lateral, Axial, or Coronal are much less frequent.

To sum up, with these previous preprocessing steps, the descriptions of selected columns are:
| Column Name       | Type        | Description                                                              | Comment/Importance                                                                 |
|-------------------|-------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| `patientid`       | Categorical | Unique identifier for each patient.                                       | Essential for patient-wise split and identifying multiple images per individual.   |
| `offset`          | Numerical   | Time offset (in days) from the first image for the patient.               | Useful for temporal modeling or disease progression tracking.                      |
| `sex`             | Categorical | Biological sex of the patient: Male (M) or Female (F).                    | Key demographic feature, potentially relevant in model bias or outcome analysis.   |
| `age`             | Numerical   | Age of the patient in years.                                              | Crucial for risk stratification and epidemiological analysis.                      |
| `finding`         | Categorical | Radiological diagnosis (e.g., COVID-19, Pneumonia, No Finding).          | Main label used for classification or grouping cases.                              |
| `RT_PCR_positive` | Categorical | RT-PCR result: Y (positive), Unclear, or missing.                         | Gold-standard COVID confirmation; helpful but incomplete.                          |
| `view`            | Categorical | View type of the image (PA, AP, Lateral, etc.).                           | Important for interpretability; some views (e.g., AP Supine) affect appearance.    |
| `date`            | Date/String | Acquisition date of the image.                                            | Can be used for time-series analysis or identifying data drift.                    |
| `folder`          | Categorical | Directory where the image is stored (mostly "images").                    | Redundant after filtering, but kept for traceability.                              |
| `filename`        | Categorical | Standardized name of the image file.                                      | Critical for linking image files to metadata.                                      |
| `clinical_notes`  | Text        | Free-text notes from the source describing patient status or findings.    | Valuable for qualitative insight or optional NLP tasks (e.g., auto-labeling).      |

##### Target Variable

In this section, we collapse the detailed `finding` labels into three clinically meaningful categories: COVID-19, Other Pneumonia, and Normal, and then summarize patient age and sex within each group.

A simplified class scheme reduces label noise from rare subtypes and focuses our analysis on the primary diagnostic distinctions of interest. Comparing demographics across these groups reveals potential biases (e.g., older patients in one group) and informs whether we need to stratify or adjust sampling during modeling.


In [ ]:
# Copy metadata
df = metadata.copy()

# Filter to the three key categories and drop the generic “todo”
valid_findings = [
    'Pneumonia/Viral/COVID-19',
    'Pneumonia', # Represents non-COVID pneumonias
    'No Finding' # Normal chest X-ray
]
df = df[df['finding'].isin(valid_findings)].reset_index(drop=True)

# Map to simplified classes
def simplify_label(finding):
    if 'COVID-19' in finding:
        return 'COVID-19'
    elif finding == 'No Finding':
        return 'Normal'
    else:
        return 'Other Pneumonia'

df['class'] = df['finding'].apply(simplify_label)


In [ ]:
age_stats = df.groupby('class')['age'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max'
)
print("Age statistics by class:\n", age_stats)


COVID-19 patients are on average 57 years old, slightly older than Other Pneumonia (51 years) and Normal cases (52 years). Age ranges overlap substantially, suggesting no extreme skew, but a modest upward shift in COVID-19.


In [ ]:
sex_counts = df.groupby(['class','sex']).size().unstack(fill_value=0)
print("Sex distribution by class:\n", sex_counts)


COVID-19 cohort is male-predominant (305 men vs. 153 women), consistent with some epidemiological reports. Other Pneumonia shows the opposite pattern in this dataset (more women than men), possibly reflecting source-specific sampling. Finally, normal cases are near-balanced.

#### 3.3.2 Image Preprocessing

Having summarized the tabular metadata, we now turn to the images themselves to gain an intuitive feel for the visual differences and potential confounders across our three classes.

Before diving into model training, we standardize our raw images so that every file:
- Has a consistent, fixed resolution (224x224 pixels), ensuring compatibility with pretrained CNN backbones.  
- Is converted to true grayscale (single channel), removing any color tints or channel inconsistencies.  
- Is saved as a JPG with a uniformly structured filename: {patientid}xray{n}.jpg where `patientid` is the original identifier and `n` counts this patient’s images in order.
- Updates the `metadata` so that the `filename` and `folder` columns now point to the new images.


In [ ]:
# Parameters
SRC_DIR   = RAW_IMAGE_DIR
OUT_DIR   = PREPROC_IMAGE_DIR
NEW_SIZE  = (224, 224)
EXT       = '.jpg'

# Create output folder if needed
os.makedirs(OUT_DIR, exist_ok=True)

# Build counters to index each patient’s images
counts = defaultdict(int)

# Iterate metadata rows, process each image
new_filenames = []
for idx, row in metadata.iterrows():
    pid     = row['patientid']
    old_fn  = row['filename']
    src_path = os.path.join(SRC_DIR, old_fn)
    
    # Skip missing files
    if not os.path.isfile(src_path):
        new_filenames.append(None)
        continue
    
    # Increment patient counter
    counts[pid] += 1
    n = counts[pid]
    
    # Build new filename
    new_fn = f"{pid}_xray_{n}{EXT}"
    dst_path = os.path.join(OUT_DIR, new_fn)
    
    # Load, convert to grayscale, resize, and save
    img = Image.open(src_path).convert('L')
    img = img.resize(NEW_SIZE, Image.BILINEAR)
    img.save(dst_path, format='JPEG')  # or 'PNG'
    
    new_filenames.append(new_fn)

# Update metadata in-place
metadata['old_filename'] = metadata['filename']
metadata['filename']     = new_filenames
metadata['folder']       = 'images_preproc'

# Drop any rows where the image was missing
metadata = metadata[metadata['filename'].notnull()].reset_index(drop=True)

# Save updated metadata
metadata.to_csv(PREPROC_METADATA_PATH, index=False)

print(f"Processed {len(metadata)} images into '{OUT_DIR}', metadata updated.")


Uniform size prevents shape mismatches and allows batch processing, grayscale focuses the model on radiographic density rather than irrelevant color artifacts, consistent naming simplifies downstream data loading and traceability, and metadata alignment ensures our train/val/test split and labels remain correct after renaming.


In [ ]:
# Show the first 10 rows of the key columns
display(metadata[['old_filename','filename','folder']].head(10))

# Quick sanity checks
print("Total rows in metadata:", len(metadata))
print("Unique patients:", metadata['patientid'].nunique())
print("Sample filenames per patient:")
print(metadata.groupby('patientid')['filename'].apply(list).head(5))


Now that we have updated our metadata with new filenames in `images_preproc/`, let’s recreate our main DataFrame (`df`) as before:


In [ ]:
# Work only with the preprocessed images
df = metadata[metadata['folder']=='images_preproc'].copy()

def simplify_label(finding):
    if 'COVID-19' in finding:
        return 'COVID-19'
    elif finding == 'No Finding':
        return 'Normal'
    else:
        return 'Other Pneumonia'

df['class'] = df['finding'].apply(simplify_label)


Examining a few prototypical images helps confirm that the model will need to learn medically meaningful patterns (e.g. ground-glass opacities in COVID-19) rather than spurious signals (e.g. hospital labels, tubes). The goal is to identify any non-pulmonary artifacts (text, lines, devices) and appreciate the overall variation in image quality and pathology presentation.


In [ ]:
# Load one example per class
examples = {}
for cls in ['COVID-19', 'Other Pneumonia', 'Normal']:
    fname = df[df['class']==cls].iloc[0]['filename']
    path = PREPROC_IMAGE_DIR / fname
    examples[cls] = Image.open(path).convert('L')

# Plot side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (cls, img) in zip(axes, examples.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(cls, fontsize=14)
    ax.axis('off')

plt.suptitle('Sample Chest X-Rays by Class', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# Original Images Visualization
samples = []
for cls in ['COVID-19', 'Other Pneumonia', 'Normal']:
    fname = df[df['class'] == cls].iloc[0]['old_filename']
    path = RAW_IMAGE_DIR / fname
    samples.append((cls, path))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (cls, img_path) in zip(axes, samples):
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Original {cls}", fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()


The COVID-19 image shows the classic diffuse, patchy opacities, whereas the Other Pneumonia case appears denser and more confluent.

Non-pulmonary artifacts (labels, tubes) are present in the Other Pneumonia and Normal examples, underscoring the need for lung-field segmentation or cropping to avoid spurious model cues. Due to this, we manually annotated a small random subset of images to quantify how often non-pulmonary artifacts appear in each class. This helps us understand potential confounders and underscores the need for preprocessing steps like lung-field segmentation.


In [ ]:
# Sample 20 distinct filenames
sample_imgs = (
    df[['filename','class']]
      .drop_duplicates(subset='filename')
      .sample(n=20, random_state=42)
      .reset_index(drop=True)
)

# Add empty columns for manual labels
sample_imgs['has_text']   = ''   # e.g. enter '1' if there is overlay text, '0' otherwise
sample_imgs['has_device'] = ''   # e.g. enter '1' if you see tubes/cables, '0' otherwise

# Save out a CSV for manual annotation
sample_imgs.to_csv('artifact_annotations_template.csv', index=False)
print("Wrote template to artifact_annotations_template.csv. Please open it,")
print("set has_text and has_device to 0 or 1 for each row, then save as artifact_annotations.csv.")


In [ ]:
# Load annotated CSV
artifact_df = pd.read_csv(ARTIFACT_ANNOTATIONS_PATH)

# Drop the annotation’s own 'class' column to avoid the suffixing collision
artifact_df = artifact_df.drop(columns=['class'])

# Merge in the true labels from df
merged = artifact_df.merge(
    df[['filename','class']], 
    on='filename', 
    how='left',
    validate='many_to_one'
)

# Check for any missing merges
missing = merged['class'].isna().sum()
print(f"Rows with no class assigned: {missing}")

# Convert your flags to numeric
merged['has_text']   = pd.to_numeric(merged['has_text'],   errors='coerce').fillna(0).astype(int)
merged['has_device'] = pd.to_numeric(merged['has_device'], errors='coerce').fillna(0).astype(int)

# Compute the per-class prevalence
summary = merged.groupby('class')[['has_text','has_device']].mean()
print("\nArtifact prevalence by class:\n", summary)

plt.figure(figsize=(6,4))
sns.heatmap(summary, annot=True, cmap='Blues', fmt=".2f")
plt.title('Proportion of Images with Text / Device Artifacts')
plt.ylabel('Class')
plt.show()


In our manually annotated subset, text artifacts are present in 70 % of COVID-19 images and 40 % of Other Pneumonia cases, while no such artifacts appear in Normal X-rays. Medical devices (e.g., tubes, catheters) are visible in 20 % of images for both COVID-19 and Other Pneumonia classes. These findings suggest that non-pulmonary visual cues are more prevalent in pathological cases and could potentially bias model predictions if not addressed through preprocessing (e.g., segmentation or cropping).

These patterns highlight two critical pre-processing needs:

1. Text Removal or Masking:  
   - Text labels are strongly correlated with disease classes and could act as spurious shortcuts for the model. We must either crop out image borders or apply in-painting/masking to eliminate overlaid text.

2. Device Artifact Handling:  
   - Tubing and other devices are genuine clinical signs of severe illness but may not generalize (e.g., they could reflect ICU setting rather than COVID-specific pathology). We should consider:  
     - Segmenting strictly to the lung fields (removing peripheral areas where devices appear).  
     - Augmenting the model’s training with device-free examples or explicitly labeling/routing device regions to avoid over-reliance.

By addressing these artifacts in our preprocessing pipeline, through lung-field segmentation, border cropping, or targeted masking, we improve our model’s focus on true radiographic features of pneumonia and avoid learning “shortcuts” that would hamper generalization to new data.

### 3.4 Exploratory Data Analysis (EDA)

This section visually explores the numeric summaries and highlight any outliers or imbalances that may warrant attention in downstream modeling (e.g., oversampling or class-weighting strategies).

#### 3.4.1 Age Distribution by Class

The boxplot below compares the age distribution across the three target classes:


In [ ]:
plt.figure(figsize=(8,6))
df.boxplot(column='age', by='class', grid=False)
plt.title('Age Distribution by Class')
plt.suptitle('')
plt.xlabel('Class')
plt.ylabel('Age (years)')
plt.show()


As seen before numerically:

- COVID-19 patients tend to be older on average, with several outliers over 85.
- Normal cases are relatively balanced but fewer in number.
- Other Pneumonia cases show broader age variation, including some younger patients.

This age skew suggests that age could act as a predictive feature, but it also raises concerns about age-related confounding in model predictions.

#### 3.4.2 Sex Distribution within Each Class

The bar chart shows the breakdown of biological sex (M, F, Unknown) by class:


In [ ]:
sex_counts.plot(kind='bar', figsize=(8,6))
plt.title('Sex Distribution within Each Class')
plt.xlabel('Class')
plt.ylabel('Number of Patients')
plt.xticks(rotation=0)
plt.legend(title='Sex')
plt.show()


- COVID-19 cases are predominantly male, consistent with epidemiological findings.
- Other Pneumonia exhibits a female-majority in this dataset, possibly due to sampling bias.
- Normal cases are sparse and roughly balanced.

#### 3.4.3 View Type Proportions

As seen before, chest X-rays can be taken from multiple anatomical perspectives:


In [ ]:
view_counts = metadata['view'].value_counts()
plt.figure(figsize=(6,4))
view_counts.plot.pie(autopct='%1.1f%%')
plt.title('Proportion of View Types (PA/AP/Lateral)')
plt.ylabel('')
plt.show()


Since view type can influence the apparent size and contrast of anatomical structures (e.g. heart, lungs), we may consider filtering or stratifying based on view or including it as a model input.

#### 3.4.4 Class Imbalance

This bar plot highlights the strong class imbalance mentioned before:


In [ ]:
# Bar plot of class counts
class_counts = df['class'].value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, 
            dodge=False, palette='pastel', legend=False)
plt.title("Number of Images per Class")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


#### 3.4.5 Images per Patient

Most patients have 1–3 X-rays, but a few contribute more. This variation confirms the need for patient-wise data splitting to avoid leakage and artificially inflated performance.


In [ ]:
# Distribution of image count per patient
img_per_patient = df['patientid'].value_counts()
plt.figure(figsize=(6,4))
sns.histplot(img_per_patient, bins=10, kde=False, color='steelblue')
plt.title("Number of Images per Patient")
plt.xlabel("Images per Patient")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()


#### 3.4.6 Word Cloud for Clinical Notes per Class (Diagnosis)

We create Word Clouds from the clinical notes for each class (COVID-19, Other Pneumonia and Normal), summarizing the most frequently occurring medical terms and observations.


In [ ]:
for c in df['class'].unique():
    text = " ".join(df[df['class'] == c]['clinical_notes'].dropna())
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud for Class: {c}")
    plt.show()


The Word Clouds reveal that clinical language varies significantly between classes. In COVID-19 cases, common terms include “fever”, “bilateral” and “positive”, all indicative of the disease's radiological and symptomatic profile. Meanwhile, Other Pneumonia entries highlight terms such as “consolidation” and “upper lobe”, often linked to bacterial or non-COVID pneumonia. For Normal, words like “normal”, “test” and “negative” dominate, suggesting these reports frequently reference absence of pathology or standard check-ups. This supports that clinical notes embed strong signals of the diagnosis—something we must account for in model interpretation to avoid data leakage.

### 3.5 Patient-Level Transformation

Until now, each row in our dataset corresponded to a single X-ray image. As observed in the earlier chart showing "images per patient", many individuals in the dataset have undergone multiple X-rays, some within days of each other. This naturally raises the question of how diagnoses evolve over time, and whether any patients show diagnostic progression or recovery.

To explore this, we transform the dataset so that each row represents a single patient, aggregating their metadata across all available X-rays. This transformation enables:

- Tracking the temporal sequence of diagnoses (e.g., `COVID-19 → Normal`)
- Identifying patients with label transitions, which may indicate improvement or mislabeling
- Preparing inputs for longitudinal or temporal modeling
- Structuring clinical notes and metadata for NLP-based patient profiling

For each patient, we store:
- Chronologically ordered image filenames and offset values
- Full diagnosis sequence (e.g., `COVID-19 → Other Pneumonia`)
- Set of unique labels observed
- Sex, age, and concatenated clinical notes


In [ ]:
# Create a new DataFrame where each row is a patient
# Helper function: get most common value or first non-null
def get_most_common_or_first(series):
    counts = series.dropna().value_counts()
    return counts.index[0] if not counts.empty else None

# Maintain patient order as in original df
patient_order = df['patientid'].drop_duplicates().tolist()

patient_records = []

for pid in patient_order:
    group = df[df['patientid'] == pid].sort_values(by='offset')
    
    filenames = group['filename'].tolist()
    offsets = group['offset'].tolist()
    classes  = group['class'].tolist()
    
    record = {
        'patientid': pid,
        'num_images': len(filenames),
        'filenames': filenames,
        'offsets': offsets,
        'diagnosis_sequence': ' → '.join(classes),
        'unique_classes': set(classes),
        'has_label_transition': len(set(classes)) > 1,
        'offset_range': max(offsets) - min(offsets) if len(offsets) > 1 else 0.0,
        'sex': get_most_common_or_first(group['sex']),
        'age': group['age'].median(),
        'clinical_notes': '\n\n'.join(group['clinical_notes'].dropna().astype(str))
    }
    
    patient_records.append(record)

df_patient = pd.DataFrame(patient_records)

# Display preview
df_patient.head()


#### 3.5.1 Label Transition Statistics


In [ ]:
# Count how many patients have a label transition
transition_counts = df_patient['has_label_transition'].value_counts()
print("Patients with label transitions:", transition_counts)

# Show percentage
total_patients = len(df_patient)
pct = transition_counts.get(True, 0) / total_patients * 100
print(f"\n{transition_counts.get(True, 0)} patients ({pct:.1f}%) experienced a class change.")

# Top 10 most common diagnosis sequences
top_sequences = df_patient['diagnosis_sequence'].value_counts().head(10)
print("\nTop diagnosis sequences:\n", top_sequences)


Out of 449 patients, only 14 (3.1 %) experienced a change in diagnosis across their X-rays.

Most patients were consistently diagnosed with a single class such as `"COVID-19"` or `"Other Pneumonia"`. A few rare transitions include:

- `"Normal → COVID-19"` — potential disease onset
- `"COVID-19 → Normal"` — possible recovery
- `"COVID-19 → Other Pneumonia"` — diagnostic uncertainty or reclassification

The bar chart below displays the 10 most frequent diagnosis sequences across patients:


In [ ]:
# Horizontal bar plot
plt.figure(figsize=(10, 5))
top_sequences.plot(kind='barh', color='orange')
plt.gca().invert_yaxis()  # Highest count on top
plt.title("Top 10 Diagnosis Sequences")
plt.xlabel("Number of Patients")
plt.ylabel("Diagnosis Progression")
plt.tight_layout()
plt.show()


#### 3.5.2 Sex Distribution Among Patients with Diagnosis Transitions

We now examine the sex distribution only among patients who changed diagnosis across their X-rays. These cases are of particular interest for clinical progression, misclassification detection or model explainability.

The stacked bar plot below shows, for each observed diagnosis sequence, how many patients were Male, Female or Unknown:

- Most transitions occur in COVID-19 patients, consistent with their over-representation in the dataset.
- Transitions such as `"COVID-19 → Normal"` or `"COVID-19 → Other Pneumonia"` are more common in males, but this may reflect dataset composition more than true clinical patterns.

This type of analysis is helpful to detect demographic bias or uncover trends in disease progression across patient subgroups.


In [ ]:
# Filter patients with diagnosis transitions
transitions_df = df_patient[df_patient['has_label_transition']]

# Group by diagnosis_sequence and sex
grouped = transitions_df.groupby(['diagnosis_sequence', 'sex']).size().unstack(fill_value=0)

# Plot
grouped.plot(kind='bar', stacked=True, figsize=(10,6), colormap='Set2')
plt.title("Sex Distribution in Patients with Diagnosis Transitions")
plt.xlabel("Diagnosis Sequence")
plt.ylabel("Number of Patients")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


#### 3.5.3 Age Distribution per Diagnosis

We create a Violin Plot showing the age distribution of patients across classes to observe potential demographic trends or biases.


In [ ]:
plt.figure(figsize=(10, 6))  # Wider plot to avoid label overlap
sns.violinplot(data=transitions_df, x='diagnosis_sequence', y='age', inner='quartile')

plt.title("Age Distribution per Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Age")
plt.xticks(rotation=30, ha='right') # Rotate x-axis labels to avoid overlap
plt.tight_layout()
plt.show()


Most sequences cluster around 60–80 years, with some paths (like “Normal → Other Pneumonia”) having more patients. It helps relate age to how diagnoses evolve over time.

### 3.6 Training with Tabular Data

While chest X-ray images provide the primary input for visual diagnosis, clinical notes accompanying each image often contain useful textual clues. These notes may reference patient history, symptoms, treatments or radiological impressions.

In this section, we explore whether we can classify X-rays into COVID-19, Other Pneumonia or Normal based only on metadata and free-text clinical notes, without using the image pixels.

#### 3.6.1 Cleaning and Preprocessing

We preprocess and vectorize the clinical notes, encode structured variables like age, sex and view type, and train a Random Forest classifier to evaluate how informative these fields are.


In [ ]:
df


In [ ]:
cols_to_drop = ['finding', 'date', 'folder', 'old_filename', 'RT_PCR_positive']
df.drop(columns=cols_to_drop, inplace=True)
print("Final features:", df.columns.tolist())


In [ ]:
# Ensure NLTK resources are downloaded
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Define lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Terms to remove to avoid label leakage
biased_terms = set([
    "covid", "covid19", "covid-19", "coronavirus", "sars",
    "pneumonia", "bacterial", "viral", "normal", "clear",
    "unremarkable", "opacity", "opacities", "bilateral",
    "diffuse", "groundglass", "consolidation", "lobar",
    "ards", "rtpcr", "rt-pcr", "swab", "nasopharyngeal",
    "positive", "negative", "diagnosed", "infected"
])

# Text cleaning function
def clean_notes(text):
    if pd.isnull(text): return ""
    text = text.lower()
    text = re.sub(r'\b\d{1,3}[-\s]?year[-\s]?old\b', '', text)
    text = re.sub(r'\b(male|female|man|woman|boy|girl)\b', '', text)
    text = re.sub(r'\b(january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{1,2}\s+\d{4}\b', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace("ground-glass", "groundglass").replace("rt-pcr", "rtpcr")
    tokens = text.split()
    clean_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens if word not in stop_words and word not in biased_terms
    ]
    return ' '.join(clean_tokens)

# Apply cleaning
df['clean_notes'] = df['clinical_notes'].apply(clean_notes)
df = df.drop(columns=['clinical_notes'])


In [ ]:
df


#### 3.6.2 Random Forest Classifier

We combine the cleaned text features with structured variables:

- Age (scaled)
- Sex (binary: M = 1, F = 0)
- View type (one-hot encoded)

These are used to train a Random Forest classifier and evaluate its ability to distinguish between the three diagnostic classes.


In [ ]:
# TF-IDF on cleaned clinical notes
vectorizer = TfidfVectorizer(max_features=3000)
X_text = vectorizer.fit_transform(df["clean_notes"])

# Encode metadata
df["sex_bin"] = df["sex"].map({"M": 1, "F": 0})
view_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
view_encoded = view_ohe.fit_transform(df[["view"]])
age_scaled = df["age"] / 100.0

# Combine metadata
X_meta = np.hstack([
    age_scaled.values.reshape(-1, 1),
    df["sex_bin"].values.reshape(-1, 1),
    view_encoded
])

# Combine text + metadata
X_full = np.hstack([X_text.toarray(), X_meta])

# Encode target variable
le = LabelEncoder()
y = le.fit_transform(df["class"])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)

# Train model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluation
y_pred = clf.predict(X_test)
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


The model achieves:

- ~90% accuracy for COVID-19 cases
- ~85% weighted overall accuracy
- Lower performance for the "Normal" class due to class imbalance

This shows that clinical notes, even when cleaned and stripped of direct class references, still contain useful signals for classification. Combined with metadata, they form a strong baseline for textual patient profiling.

#### 3.6.3 Feature Importance


In [ ]:
# Combine feature names
tfidf_feature_names = vectorizer.get_feature_names_out()
meta_feature_names = ["age", "sex_bin"] + list(view_ohe.get_feature_names_out(["view"]))
all_feature_names = list(tfidf_feature_names) + meta_feature_names

# Get importances
importances = clf.feature_importances_
indices = np.argsort(importances)[-20:][::-1]
top_features = [all_feature_names[i] for i in indices]
top_importances = importances[indices]

# Plot top 20 features
plt.figure(figsize=(10, 6))
plt.barh(top_features[::-1], top_importances[::-1])
plt.xlabel("Feature Importance")
plt.title("Top 20 Most Influential Features (Random Forest)")
plt.tight_layout()
plt.show()


The horizontal bar chart above shows the top 20 features (from both clinical notes and structured metadata) that most influenced the predictions of the Random Forest classifier trained to distinguish between COVID-19, Other Pneumonia, and Normal cases.

- `age` is the most important feature by far. This likely reflects clinical reality, COVID-19 and pneumonia cases are more common in older individuals.
- `view_AP Supine`, `view_PA` and `sex_bin` also appear among the top features, showing that demographic and technical metadata strongly contribute to prediction.
- Words like `right`, `lobe`, `thickening`, `oxygen` and `fever` are derived from cleaned clinical notes and are medically relevant. These likely reflect radiological findings or symptoms described by clinicians, e.g., “right lower lobe thickening” or “oxygen support required”.
- The model finds patterns in view types. For example, `AP Supine` is more common in ICU or bedridden patients and may correlate with severe disease.

#### 3.6.4 XGBoost

To explore an alternative model that works well with explainability techniques, we train an XGBoost classifier using both:
- Structured metadata (age, sex, view type), and
- TF-IDF vectorized clinical notes

This model allows us to combine free-text and tabular inputs into a single prediction pipeline.


In [ ]:
# TF-IDF on clean clinical notes
vectorizer = TfidfVectorizer(max_features=3000)
X_text = vectorizer.fit_transform(df["clean_notes"])

# Encode sex and view
df["sex_bin"] = df["sex"].map({"M": 1, "F": 0})
view_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
view_encoded = view_ohe.fit_transform(df[["view"]])
age_scaled = df["age"] / 100.0

# Structured metadata
X_meta = np.hstack([
    age_scaled.values.reshape(-1, 1),
    df["sex_bin"].values.reshape(-1, 1),
    view_encoded
])

# Final feature matrix
X_full = np.hstack([X_text.toarray(), X_meta])

# Encode target
le = LabelEncoder()
y = le.fit_transform(df["class"])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)

# Train XGBoost model
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train)


In [ ]:
# Evaluation
y_pred = xgb_model.predict(X_test)
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


#### 3.6.5 SHAP and LIME

##### SHAP: Global Feature Impact

To analyze global model behavior, we use SHAP (SHapley Additive Explanations) with the structured metadata features only (e.g., age, sex, view type), since they are the most interpretable.

We extract the 10 most important structured features and visualize how much they contribute to the model’s predictions on average.


In [ ]:
# Use only structured metadata (last columns)
meta_feature_names = ["age", "sex_bin"] + list(view_ohe.get_feature_names_out(["view"]))
X_train_meta = X_train[:, -len(meta_feature_names):]

# Get top 10 features by importance
importances_meta = xgb_model.feature_importances_[-len(meta_feature_names):]
top_meta_idx = np.argsort(importances_meta)[-10:]
top_meta_names = [meta_feature_names[i] for i in top_meta_idx]
X_train_meta_top10 = X_train_meta[:, top_meta_idx]

# SHAP explainer and values
explainer = shap.TreeExplainer(xgb_model)
shap_values_meta = explainer.shap_values(X_train)
shap_values_meta_top10 = shap_values_meta[:, -len(meta_feature_names):][:, top_meta_idx]

# SHAP summary plot (bar)
shap.summary_plot(
    shap_values_meta_top10,
    features=X_train_meta_top10,
    feature_names=top_meta_names,
    plot_type="bar"
)


The resulting bar chart shows `age`, `view_PA` and `view_AP Supine` as the most influential features across the dataset, with class-specific SHAP contributions.

##### LIME: Local Explanation for a Single Prediction

To explain individual predictions, we apply LIME (Local Interpretable Model-Agnostic Explanations) to a single test instance.

We limit the input to structured features only (to keep explanations concise) and avoid automatic binning for safety.


In [ ]:
# Create LIME explainer
lime_explainer = LimeTabularExplainer(
    training_data=X_train_meta,
    feature_names=meta_feature_names,
    class_names=le.classes_,
    mode='classification',
    discretize_continuous=False
)

# Select a test sample to explain
instance_idx = 5
sample = X_test[instance_idx, -len(meta_feature_names):]

# Define a wrapper predict function (metadata only)
predict_fn = lambda x: xgb_model.predict_proba(np.hstack([
    np.zeros((x.shape[0], X_text.shape[1])),  # dummy for TF-IDF columns
    x
]))

# Generate LIME explanation
lime_exp = lime_explainer.explain_instance(
    data_row=sample,
    predict_fn=predict_fn,
    num_features=7
)

# Show in notebook
lime_exp.show_in_notebook(show_table=True)


The LIME panel shows the predicted class and a list of features with their values and contributions. For this patient, `view_AP Supine`, `age` and absence of `view_PA` were the strongest predictors of the model’s COVID-19 diagnosis.

### 3.7 Training with Images

In this section, we train a Convolutional Neural Network (CNN) to classify chest X-ray images directly. We also use Grad-CAM to visualize which regions of the images are most important for the model's predictions.

#### 3.7.1 Data Preparation

We begin by preparing a simplified version of the metadata DataFrame and defining a PyTorch Dataset class to load and preprocess the chest X-ray images.


In [ ]:
# Copy relevant columns
df_final = metadata.copy()

# Drop unused metadata columns
cols_to_drop = ['date', 'folder', 'old_filename', 'RT_PCR_positive', 'clinical_notes']
df_final.drop(columns=cols_to_drop, inplace=True)

# Assign class labels
df_final['class'] = df_final['finding'].apply(simplify_label)
df_final = df_final.drop(columns=['finding'])

# Rename for consistency
df = df_final


The `ChestXrayDataset` class reads images from disk, converts them to RGB and maps class labels to integer values. Images are resized to 224×224 and normalized.


In [ ]:
# Dataset class
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.label_map = {'COVID-19': 0, 'Other Pneumonia': 1, 'Normal': 2}
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        label = self.label_map[row['class']]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Image transform pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # Normalize RGB channels
])


#### 3.7.2 Train/Val/Test Split

We perform a patient-wise stratified split to ensure class balance across train, validation, and test sets. This is crucial given the imbalance in “Normal” samples.


In [ ]:
# Stratified split to maintain class balance
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['class'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['class'], random_state=42)

# Load datasets
train_dataset = ChestXrayDataset(train_df, PREPROC_IMAGE_DIR, transform)
val_dataset   = ChestXrayDataset(val_df,   PREPROC_IMAGE_DIR, transform)
test_dataset  = ChestXrayDataset(test_df,  PREPROC_IMAGE_DIR, transform)


#### 3.7.3 Model Definition: Simple CNN

The CNN consists of two convolutional layers followed by max pooling and two fully connected layers. It’s a lightweight architecture suitable for quick prototyping on a limited dataset.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (B, 16, 112, 112)
        x = self.pool(F.relu(self.conv2(x)))  # (B, 32, 56, 56)
        x = x.view(-1, 32 * 56 * 56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


#### 3.7.4 Training and Evaluation

The model is trained over 10 epochs using cross-entropy loss and Adam optimizer. Accuracy and loss are tracked for both training and validation sets.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=3).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return running_loss / len(loader), accuracy, all_preds, all_labels


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} - "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


The training accuracy improves steadily, reaching ~86% by epoch 10. Validation accuracy fluctuates more, peaking at 72%, which may suggest some overfitting. Loss values also suggest the model struggles more on the validation set, likely due to the small number of “Normal” cases.


In [ ]:
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion)

print("\nTest Accuracy:", test_acc)
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=['COVID-19', 'Other Pneumonia', 'Normal']))
print("Confusion Matrix:\n")
print(confusion_matrix(labels, preds))


The model achieves 66.15% accuracy on the test set. It performs well for COVID-19 and reasonably for Other Pneumonia. However, the Normal class is never predicted, leading to zero precision/recall. This highlights a major limitation of class imbalance and the difficulty of distinguishing healthy cases with so few examples.

#### 3.7.5 Grad-CAM for CNN Interpretability

We now apply Grad-CAM to visualize which parts of the X-ray influenced the CNN’s decision.


In [ ]:
# Global variables for Grad-CAM hooks
activations, gradients = None, None

def forward_hook(module, input, output):
    global activations
    activations = output

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

# Register hooks
target_layer = model.conv2
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)


In [ ]:
def generate_gradcam(model, image_tensor, class_idx=None):
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    output = model(image_tensor)
    
    if class_idx is None:
        class_idx = output.argmax().item()

    model.zero_grad()
    class_score = output[0, class_idx]
    class_score.backward()

    pooled_grad = torch.mean(gradients, dim=(0, 2, 3))
    activation = activations[0]
    
    for i in range(len(pooled_grad)):
        activation[i, :, :] *= pooled_grad[i]
        
    heatmap = activation.mean(dim=0).cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap)

    return heatmap


In [ ]:
def show_gradcam_overlay(image_tensor, heatmap, alpha=0.4):

    image = image_tensor.permute(1, 2, 0).cpu().numpy()
    image = (image * 0.5 + 0.5)
    image = np.uint8(255 * image)

    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    colormap = cm.get_cmap('jet') if hasattr(cm, 'get_cmap') else cm.jet
    heatmap_rgb = np.uint8(255 * colormap(heatmap_resized)[:, :, :3])

    overlay = cv2.addWeighted(image, 1 - alpha, heatmap_rgb, alpha, 0)

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].imshow(image)
    axs[0].set_title("Original Image")
    axs[0].axis('off')

    im = axs[1].imshow(overlay)
    axs[1].set_title("Grad-CAM Overlay")
    axs[1].axis('off')
    cbar = fig.colorbar(cm.ScalarMappable(cmap='jet'), ax=axs[1], fraction=0.046, pad=0.04)
    cbar.set_label("Attention Intensity")

    plt.tight_layout()
    plt.show()


In [ ]:
# Example Grad-CAM
sample_img, _ = test_dataset[1]
heatmap = generate_gradcam(model, sample_img)
show_gradcam_overlay(sample_img, heatmap)


The Grad-CAM visualization shows that the model attends strongly to lung regions with diffuse opacities, areas typically affected in COVID-19. The attention map aligns well with clinical expectations, indicating that the CNN is learning meaningful radiological patterns rather than spurious signals. However, without preprocessing to remove artifacts (e.g., tubes or overlays), some attention may still fall outside the lungs.

However, in the second case (shown below), the attention map appears much less focused, diffused over lines and tubes rather than lung structures. This may indicate that the model is relying on non-pulmonary artifacts (e.g., medical devices) as shortcuts for class prediction.


In [ ]:
sample_img, _ = test_dataset[58]
heatmap = generate_gradcam(model, sample_img)
show_gradcam_overlay(sample_img, heatmap)


While some Grad-CAM results may appear clinically relevant, this can be misleading, especially in cherry-picked cases. Broader evaluation across more diverse samples reveals that attention is often misdirected or ambiguous, emphasizing the need for further preprocessing or bias mitigation.

### 3.8 Explainability with Pretrained X-Ray Models

Until now, we trained simple classifiers using tabular data and custom CNNs. However, due to the absence of GPU acceleration and the goal of analyzing model attention, not improving state-of-the-art performance, we turn to pretrained models.

**Why a pretrained model?**
- Training deep models (e.g., DenseNet121) from scratch requires significant time and computational power.
- Our focus is explainability, not necessarily accuracy.
- We use 'torchxrayvision', a Python library providing pretrained models for chest X-rays, trained on large-scale datasets like NIH and CheXpert.
- It includes class labels like Effusion, Lung Opacity, Cardiomegaly, etc.

We will apply Grad-CAM to understand which regions of the input X-ray influenced the prediction most.

#### 3.8.1 Grad-CAM Visualization with TorchXRayVision


In [ ]:
def visualize_gradcam_xrv(img_path, target_class="Lung Opacity", alpha=0.4):
    # Load pretrained model
    model = xrv.models.DenseNet(weights="densenet121-res224-all")
    model.eval()

    # Image transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # Normalization required by model
    ])

    # Load and prepare image
    img = Image.open(img_path).convert("L")
    input_tensor = transform(img).unsqueeze(0)
    input_tensor.requires_grad_()

    # Get class index
    class_idx = model.pathologies.index(target_class)

    # Define Grad-CAM
    cam_extractor = GradCAM(model=model, target_layer="features.norm5")
    output = model(input_tensor)
    activation_map = cam_extractor(class_idx, output)

    # Prepare original image
    original_img = input_tensor.squeeze().detach().numpy()
    original_img = (original_img * 0.5 + 0.5) * 255
    original_img = np.uint8(original_img)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_GRAY2RGB)

    # Create Grad-CAM overlay
    heatmap = activation_map[0].squeeze().detach().numpy()
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)

    print("Top prediction:", model.pathologies[class_idx], "with score", output[0, class_idx].item())

    print("\nFull output:")
    for i, label in enumerate(model.pathologies):
        print(f"{label:25s} {output[0, i].item():.4f}")

    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].imshow(original_img)
    axs[0].set_title("Original Image")
    axs[0].axis("off")

    vmin, vmax = 100, 255
    im = axs[1].imshow(overlay, cmap='jet', vmin=vmin, vmax=vmax)
    axs[1].set_title(f"Grad-CAM: {target_class}")
    axs[1].axis("off")

    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap='jet'), ax=axs[1], fraction=0.046, pad=0.04)
    cbar.set_label("Attention Intensity")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_gradcam_xrv(PREPROC_IMAGE_DIR / "222_xray_2.jpg", target_class="Lung Opacity")


#### 3.8.2 Grad-CAM Across Layers

To better understand how feature extraction evolves through the model, we visualize Grad-CAM at various internal layers.


In [ ]:
def visualize_gradcam_progressive(img_path, target_class="Lung Opacity", alpha=0.4):
    model = xrv.models.DenseNet(weights="densenet121-res224-all")
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    img = Image.open(img_path).convert("L")
    input_tensor = transform(img).unsqueeze(0)
    input_tensor.requires_grad_()

    class_idx = model.pathologies.index(target_class)

    # Target layers at various depths
    layer_names = [
        "features.conv0",
        "features.denseblock1.denselayer6.conv2",
        "features.denseblock2.denselayer12.conv2",
        "features.denseblock3.denselayer24.conv2",
        "features.denseblock4.denselayer16.conv2",
        "features.norm5"
    ]

    original_img = input_tensor.squeeze().detach().numpy()
    original_img = (original_img * 0.5 + 0.5) * 255
    original_img = np.uint8(original_img)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_GRAY2RGB)

    fig, axs = plt.subplots(1, len(layer_names), figsize=(4 * len(layer_names), 4))

    for i, layer in enumerate(layer_names):
        cam_extractor = GradCAM(model=model, target_layer=layer)
        output = model(input_tensor)
        activation_map = cam_extractor(class_idx, output)

        heatmap = activation_map[0].squeeze().detach().numpy()
        heatmap_resized = cv2.resize(heatmap, (224, 224))
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)

        axs[i].imshow(overlay)
        axs[i].set_title(layer.split('.')[-2] if '.' in layer else layer)
        axs[i].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_gradcam_progressive(PREPROC_IMAGE_DIR / "222_xray_2.jpg", target_class="Lung Opacity")


While using a pretrained model allows us to apply Grad-CAM without training, the results are not always reliable. Some activation maps highlight irrelevant areas or borders, likely due to input domain shift (our data vs. training data).

The torchxrayvision model was trained on NIH/CheXpert datasets, which differ from COVID-19-specific data (lighting, devices, population). This approach is extremely useful for interpretability, but must be used with caution when applied to new domains.

## 4. Conclusions

This project explored multimodal COVID-19 detection using chest X-ray images and clinical metadata, combining explainable machine learning (XAI) techniques with domain-specific knowledge. We implemented both custom models and pretrained networks to highlight the importance of feature importance analysis and model interpretability in medical diagnostics.

Our Random Forest and XGBoost models, enriched with TF-IDF clinical notes and metadata, yielded promising results, with age and view type emerging as key predictors. SHAP and LIME successfully revealed how different structured features influenced model decisions.

We further trained a lightweight CNN to classify raw chest X-rays and applied Grad-CAM to analyze attention regions. Although some visualizations aligned with clinical expectations, others suffered from artifacts or overfitting, underscoring the challenge of interpretability in medical imaging.

Finally, to overcome computational limits and gain richer visual insights, we integrated a pretrained DenseNet121 model from TorchXRayVision. This allowed us to produce Grad-CAM overlays and layer-wise progression maps, although interpretability remained inconsistent due to domain shift and input artifacts.

Overall, the project demonstrates that explainability tools are essential, but must be critically assessed, when applied to complex multimodal medical data.

## 5. Future Work

This project opens up several interesting directions for future research:

- Patient-level representation: Continuing the approach where each row represents a patient (rather than a single image) would allow for modeling disease progression and temporal patterns more effectively.
- Data augmentation: Applying controlled augmentations (e.g., image rotations or horizontal flips) could reduce overfitting while maintaining clinical realism.
- Lung field segmentation: Introducing a preprocessing step to segment lungs (e.g., with a U-Net) would help remove distracting artifacts like text or medical devices, improving model robustness.
- Deeper explainability: Exploring advanced XAI methods (e.g., Grad-CAM++, Integrated Gradients) or progressive visualization across multiple images per patient could yield richer insights.
- Multimodal models: Combining images, metadata and clinical notes into a single model would better reflect real-world diagnostic workflows and potentially improve performance.


## 6. Demo


In [ ]:
# Load your trained model and encoders,
# clf = Already loaded RandomForestClassifier,
# vectorizer =  Already loaded TfidfVectorizer,
le_sex = LabelEncoder()
le_sex.fit(["M", "F", "Unknown"])

# Manual Input (user-defined)
input_age = 24
input_sex = "M"
input_text = "dry cough and bilateral opacity"

# Preprocessing Steps
# Transform clinical notes using the trained vectorizer (TF-IDF, CountVectorizer...)
X_text = vectorizer.transform([input_text]).toarray()

# Encode sex using trained LabelEncoder
X_sex = le_sex.transform([input_sex])

# Concatenate all features: [age, encoded_sex, vectorized_text]
X_input = np.concatenate(([input_age], X_sex, X_text[0]))

# Pad or truncate input to match model's expected number of features
if len(X_input) < clf.n_features_in_:
    X_input = np.pad(X_input, (0, clf.n_features_in_ - len(X_input)), constant_values=0)
elif len(X_input) > clf.n_features_in_:
    X_input = X_input[:clf.n_features_in_]

# Make Prediction
rf_probs = clf.predict_proba(X_input.reshape(1, -1))[0]
rf_labels = clf.classes_

# Human-readable Class Mapping
label_map = {
    0: "COVID-19",
    1: "Normal",
    2: "Other Pneumonia"
}

# Display Input and Predictions
print("Input Summary:")
print(f"  Age: {input_age}")
print(f"  Sex: {input_sex}")
print(f"  Clinical Notes: {input_text}\n")

print("Random Forest Prediction Probabilities:")
for label, prob in zip(rf_labels, rf_probs):
    readable_label = label_map[label]
    print(f"  {readable_label}: {prob:.2%}")
